# Results analysis

Turns the aggregate result files into the README results table, the report's LaTeX
tables and its figures, and shows them here. The work is done by
`scripts/make_report_assets.py`, so the same assets can be rebuilt without the notebook:

```bash
python scripts/make_report_assets.py results --run gold-explain
```

It reads the files written by the evaluation scripts:

| File | Written by |
| --- | --- |
| `eval/results/metrics_gold-explain.json` | `scripts/compute_metrics.py` |
| `outputs/human_eval/ratings_rater1.csv` and `key.csv` | the first rater, and `scripts/make_rating_sheet.py` |
| `outputs/human_eval/ratings_llm_judge.csv` | `scripts/llm_judge.py` |
| `eval/results/agreement_raters.md`, `agreement_judge.md` | `scripts/agreement.py` |

Only the rating columns of the sheets are read, never the sentences or the feedback.
Missing files are skipped; with none of them nothing is written, and the cells below
say so. Intervals are 95% percentile bootstrap intervals over gold edits (automatic
metrics) or rated items (ratings); differences between systems are paired by edit.

In [ ]:
import os
import sys
from pathlib import Path

from IPython.display import Image, Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)  # the script's default paths are relative to the repository root
sys.path.insert(0, "scripts")

In [ ]:
import make_report_assets as assets

RUN = "gold-explain"
written = assets.main(["results", "--run", RUN])

In [ ]:
def show(name: str) -> None:
    """Display a figure written by the script, or say that it is not available."""
    path = Path("report/figures") / name
    if path in written:
        display(Image(filename=path))
    else:
        print(f"{name}: not available (its result files are missing)")

## Main results

Systems A to D and the teacher reference R. Automatic metrics are on the 300 gold edits;
ratings are on the rubric's 0-2 scale. R is not in the human evaluation. This table is
`eval/results/results_table.md`, for the README; the report's version is
`report/tables/results_main.tex`.

In [ ]:
table = Path("eval/results/results_table.md")
if table in written:
    display(Markdown(table.read_text(encoding="utf-8")))
else:
    print("No results table: no result files are available yet.")

## Paired differences

B-A is the effect of grounding alone, C-A of fine-tuning alone, D-C of grounding a tuned
model and D-B of fine-tuning a grounded model. An interval that excludes zero is a
difference that the 95% paired bootstrap supports.

In [ ]:
show("paired_differences_automatic.png")
show("paired_differences_human.png")

## By CEFR level and error category

Levels and categories with fewer than 10 edits (automatic metrics) or 10 rated items per
system (ratings) are left out; their intervals would be too wide to read.

In [ ]:
for name in ("results_by_level.png", "results_by_category.png"):
    show(name)

In [ ]:
for name in ("human_by_level.png", "human_by_category.png"):
    show(name)

## LLM judge against the human ratings

Agreement of the second rater and of the LLM judge with the first rater (Cohen's kappa,
and quadratic-weighted kappa for the three-point dimensions). The second rater's
agreement is the reference point: the judge can be used in place of a teacher only where
it agrees with the first rater about as well as a second teacher does. The second chart
checks whether the judge ranks the systems as the first rater does.

In [ ]:
show("judge_agreement.png")
show("judge_vs_rater_means.png")